# Projeto — Fundamentos da Descoberta de Dados

Notebook criado para o projeto do módulo. Este notebook contém a análise completa, resolução dos exercícios e explicações passo a passo em linguagem simples.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

plt.style.use('seaborn')
%matplotlib inline

In [ ]:
# Carregar os dados

df = pd.read_csv(r'MODULO7_PROJETOFINAL_BASE_SUPERMERCADO - MODULO7_PROJETOFINAL_BASE_SUPERMERCADO (1).csv', sep=';')
display(df.head(10))

# Mostrar informações gerais
print('\nInformações gerais:')
df.info()

print('\nEstatísticas descritivas (numéricas):')
df.describe(include='all')

## Limpeza e padronização das colunas

In [ ]:
# Padronizar nomes de colunas: tirar espaços e colocar minúsculas
original_cols = df.columns.tolist()
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
print('Colunas originais:', original_cols)
print('Colunas padronizadas:', df.columns.tolist())

df.head(3)

## Verificar valores faltantes e tipos de dados

In [ ]:
# Valores faltantes por coluna
print(df.isna().sum())

for col in ['preco_normal','preco_desconto','preco_anterior','desconto']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(',','.'), errors='coerce')

print('\nTipos após conversão:')
print(df.dtypes)

df.head(3)

## Exercício 1 — Média e mediana do preco_normal por categoria


In [ ]:
 
if 'categoria' in df.columns and 'preco_normal' in df.columns:
    resumo = df.groupby('categoria')['preco_normal'].agg(['mean','median','count']).reset_index()
    resumo = resumo.sort_values('mean', ascending=False)
    display(resumo)
else:
    print('Coluna categoria ou preco_normal não encontrada.')

if 'categoria' in df.columns and 'preco_normal' in df.columns:
    resumo['comparacao'] = resumo['mean'] - resumo['median']
    resumo['maior_ou_menor'] = resumo['comparacao'].apply(lambda x: 'media>mediana' if x>0 else ('media<mediana' if x<0 else 'media=mediana'))
    display(resumo[['categoria','mean','median','maior_ou_menor']].head(20))

## Exercício 2 — Desvio padrão por categoria


In [ ]:

if 'categoria' in df.columns and 'preco_normal' in df.columns:
    desv = df.groupby('categoria')['preco_normal'].agg(['std','mean','median','count']).reset_index()
    desv = desv.sort_values('std', ascending=False)
    display(desv.head(10))
else:
    print('Coluna categoria ou preco_normal não encontrada.')

if 'categoria' in df.columns:
    maior_desvio = desv.iloc[0]['categoria']
    print(f"Categoria com maior desvio padrão: {maior_desvio}")
    print('Verifique se a média é maior ou menor que a mediana:')
    display(desv[desv['categoria']==maior_desvio])

## Exercício 3 — Boxplot do preco_normal para a categoria com maior desvio padrão

In [ ]:

import matplotlib.pyplot as plt

if 'categoria' in df.columns and 'preco_normal' in df.columns:
    top_cat = desv.iloc[0]['categoria']
    dados_cat = df[df['categoria']==top_cat]['preco_normal'].dropna()
    plt.figure(figsize=(6,4))
    plt.boxplot(dados_cat)
    plt.title(f'Boxplot Preco_Normal — Categoria: {top_cat}')
    plt.ylabel('Preço Normal')
    plt.show()
    print('Número de pontos na categoria:', dados_cat.shape[0])
else:
    print('Coluna categoria ou preco_normal não encontrada.')

## Exercício 4 — Gráfico de barras com a média de desconto por categoria

In [ ]:

if 'categoria' in df.columns and 'desconto' in df.columns:
    media_desconto = df.groupby('categoria')['desconto'].mean().reset_index().sort_values('desconto', ascending=False)
    # matplotlib
    plt.figure(figsize=(10,6))
    plt.bar(media_desconto['categoria'], media_desconto['desconto'])
    plt.xticks(rotation=90)
    plt.title('Média de desconto por categoria')
    plt.ylabel('Desconto médio')
    plt.xlabel('Categoria')
    plt.show()
    display(media_desconto)
else:
    print('Coluna categoria ou desconto não encontrada.')

## Exercício 5 — Gráfico interativo agrupando por categoria e marca com média de desconto

In [ ]:

if 'categoria' in df.columns and 'marca' in df.columns and 'desconto' in df.columns:
    grp = df.groupby(['categoria','marca'])['desconto'].mean().reset_index().sort_values('desconto', ascending=False)
    grp['tamanho'] = (grp['desconto'] - grp['desconto'].min()) + 0.1
    fig = px.scatter(grp, x='categoria', y='marca', size='tamanho', color='desconto', hover_data=['desconto'], title='Média de desconto por categoria e marca')
    fig.update_layout(yaxis={'categoryorder':'total descending'}, xaxis={'tickangle':-45})
    fig.show()
    display(grp.head(30))
else:
    print('Colunas necessárias (categoria, marca, desconto) não encontradas.')